<a href="https://colab.research.google.com/github/SahanRachintha/AI_Legal/blob/main/ANN_LEGAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import io
import os

In [ ]:
!pip install pdfplumber

In [ ]:
import pdfplumber

In [ ]:
dataset=load_dataset("theatticusproject/cuad", verification_mode="no_checks")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/714 [00:00<?, ?it/s]

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pdf'],
        num_rows: 511
    })
})


In [ ]:
sample=dataset["train"][0]["pdf"]
print(dir(dataset["train"][0]["pdf"]))

['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', 'annots', 'cached_properties', 'chars', 'close', 'curve_edges', 'curves', 'doc', 'edges', 'flush_cache', 'horizontal_edges', 'hyperlinks', 'images', 'laparams', 'lines', 'metadata', 'objects', 'open', 'pages', 'pages_to_parse', 'password', 'path', 'raise_unicode_errors', 'rect_edges', 'rects', 'rsrcmgr', 'stream', 'stream_is_external', 'structure_tree', 'textboxhorizontals', 'textboxverticals', 'textlinehorizontals', 'textlineverticals', 'to_csv', 'to_dict', 'to_json', 'unicode_norm', 'vertical_edges']


In [ ]:
contract_texts = []

for i in range(len(dataset["train"])):
    pdf=dataset["train"][i]["pdf"]

    full_texts=""

    for page in pdf.pages:
        page_text=page.extract_text()

        full_texts += page_text+"\n"

    contract_texts.append(full_texts)


print("Total number of contracts extracted:", len(contract_texts))



Total number of contracts extracted: 511


In [ ]:
import re

def clean_contract_text(text):
  text = re.sub(r'-\s*\d+\s*-', '', text)
  text = re.sub(r'page\s+\d+\s+of\s+\d+', '', text, flags=re.IGNORECASE)

  text = re.sub(r'\n{3,}', '\n\n', text)
  text = re.sub(r'\s{3,}', ' ', text)

  lines = text.split('\n')
  lines = [l.strip() for l in lines if len(l.strip()) > 20]
  text = '\n'.join(lines)

  text = re.sub(r'[^\w\s\.\,\;\:\(\)\-\'\"\§\%]', ' ', text)
  return text.strip()

contract_texts_clean = [clean_contract_text(t) for t in contract_texts]
print("Sample cleaned text:")
print(contract_texts_clean[0][:500])

Sample cleaned text:
Datasheet for Contract Understanding Atticus Dataset (CUAD)
A. Who created the dataset (e.g., which team, research group) and on behalf of which entity (e.g. company,
institution, organization) 
The Atticus Project is a non-profit organization whose mission is to harness the power of AI to accelerate
accurate and efficient contract review. The Atticus Project started as a grassroots movement by experienced
lawyers in public companies and leading law firms aiming to achieve high-quality, low-cost


In [ ]:
LEGAL_MARKERS=[r"\bhereby\b", r"\bwhereas\b", r"\bagreement\b",
    r"\bshall\b", r"\bparties\b", r"\bexecuted\b",
    r"\bin witness whereof\b", r"\bterms and conditions\b"]

def is_valid_contract(text):
    if len(text.split())<200:
      return False,"Too Short"

    text_lower=text.lower()

    marker_hits = sum(1 for p in LEGAL_MARKERS
                     if re.search(p, text_lower))
    if marker_hits < 2:
        return False, "no_legal_language"

    return True, "valid"

valid_contracts = []
for text in contract_texts_clean:
    is_valid, reason = is_valid_contract(text)
    if is_valid:
        valid_contracts.append(text)

print(f"Original:  {len(contract_texts)}")
print(f"Valid:     {len(valid_contracts)}")
print(f"Removed:   {len(contract_texts) - len(valid_contracts)}")


Original:  511
Valid:     501
Removed:   10


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def remove_duplicates(texts, threshold=0.95):

    print("Checking for duplicates...")

    seen = set()
    unique_texts = []
    exact_dupes = 0

    for text in texts:
        fingerprint = text[:200].strip().lower()
        if fingerprint not in seen:
            seen.add(fingerprint)
            unique_texts.append(text)
        else:
            exact_dupes += 1

    print(f"  Exact duplicates removed: {exact_dupes}")

    vectorizer = TfidfVectorizer(max_features=1000)
    tfidf_matrix = vectorizer.fit_transform(
        [t[:1000] for t in unique_texts]  # use first 1000 chars
    )

    similarity_matrix = cosine_similarity(tfidf_matrix)


    to_remove = set()
    for i in range(len(unique_texts)):
        if i in to_remove:
            continue
        for j in range(i+1, len(unique_texts)):
            if similarity_matrix[i][j] >= threshold:
                to_remove.add(j)

    final_texts = [t for i, t in enumerate(unique_texts)
                   if i not in to_remove]

    print(f"  Near-duplicates removed:  {len(to_remove)}")
    print(f"  Original:  {len(texts)}")
    print(f"  After dedup: {len(final_texts)}")

    return final_texts

# Apply
valid_contracts = remove_duplicates(valid_contracts)

Checking for duplicates...
  Exact duplicates removed: 9
  Near-duplicates removed:  3
  Original:  501
  After dedup: 489


In [ ]:
from transformers import pipeline
import torch

# Load model — this is the legal-specific version
classifier = pipeline(
    "zero-shot-classification",
    model="casehold/legalbert",          # Legal domain BERT
    device=0 if torch.cuda.is_available() else -1
)

CONTRACT_TYPES = [
    "non-disclosure agreement",
    "employment contract",
    "service agreement",
    "lease agreement",
    "vendor agreement",
    "license agreement"
]

def label_with_legalbert(text):
    """
    Use first 512 chars — covers title + opening clauses
    which is where contract type is most clearly stated
    """
    snippet = text[:512]

    result = classifier(
        snippet,
        candidate_labels=CONTRACT_TYPES,
        hypothesis_template="This document is a {}."
    )

    return {
        "label":      result["labels"][0],
        "confidence": round(result["scores"][0], 3),
        "all_scores": dict(zip(result["labels"],
                           [round(s,3) for s in result["scores"]]))
    }

# Apply to all valid contracts
print("Labeling contracts with LegalBERT...")
labels = []
for i, text in enumerate(valid_contracts):
    result = label_with_legalbert(text)
    labels.append(result)
    if i % 50 == 0:
        print(f"  Progress: {i}/{len(valid_contracts)}")

print("Done!")

config.json:   0%|          | 0.00/740 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/441M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: casehold/legalbert
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect id

tokenizer_config.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Failed to determine 'entailment' label id from the label2id mapping in the model config. Setting to -1. Define a descriptive label2id mapping in the model config to ensure correct outputs.


Labeling contracts with LegalBERT...
  Progress: 0/489
  Progress: 50/489
  Progress: 100/489
  Progress: 150/489
  Progress: 200/489
  Progress: 250/489
  Progress: 300/489
  Progress: 350/489
  Progress: 400/489
  Progress: 450/489
Done!


In [ ]:
import pandas as pd

df = pd.DataFrame({
    "contract_id": range(len(valid_contracts)),
    "text":        valid_contracts,
    "label":       [l["label"] for l in labels],
    "confidence":  [l["confidence"] for l in labels],
    "word_count":  [len(t.split()) for t in valid_contracts]
})

# Separate high vs low confidence
high_conf = df[df["confidence"] >= 0.60]
low_conf  = df[df["confidence"] <  0.60]

print(f"High confidence (auto-accepted): {len(high_conf)}")
print(f"Low confidence (needs review):   {len(low_conf)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())

# Save — you never want to re-run LegalBERT labeling
df.to_csv("contracts_labeled.csv", index=False)
print("\nSaved to contracts_labeled.csv")

High confidence (auto-accepted): 0
Low confidence (needs review):   489

Label distribution:
label
lease agreement             243
employment contract         139
non-disclosure agreement     45
service agreement            25
vendor agreement             23
license agreement            14
Name: count, dtype: int64

Saved to contracts_labeled.csv


In [ ]:
import re
import unicodedata

def normalize_text(text):
    """
    Standard text normalization for legal NLP
    """

    # 1. Unicode normalization — fixes encoding issues from PDFs
    # e.g., converts â€™ back to '
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")

    # 2. Lowercase
    text = text.lower()

    # 3. Expand common legal abbreviations
    legal_abbreviations = {
        r"\bagmt\b":    "agreement",
        r"\bco\.\b":    "company",
        r"\bcorp\.\b":  "corporation",
        r"\binc\.\b":   "incorporated",
        r"\bltd\.\b":   "limited",
        r"\bllc\.\b":   "limited liability company",
        r"\bno\.\b":    "number",
        r"\bsec\.\b":   "section",
        r"\bart\.\b":   "article",
        r"\bexh\.\b":   "exhibit",
        r"\bsch\.\b":   "schedule",
        r"\bvs\.\b":    "versus",
        r"\betc\.\b":   "et cetera"
    }
    for pattern, replacement in legal_abbreviations.items():
        text = re.sub(pattern, replacement, text)

    # 4. Normalize whitespace
    text = re.sub(r'\s+', ' ', text)

    # 5. Normalize punctuation
    text = re.sub(r'[''`]', "'", text)       # smart quotes → straight
    text = re.sub(r'[""]', '"', text)         # smart double quotes
    text = re.sub(r'\.{2,}', '.', text)       # multiple dots → one
    text = re.sub(r'\-{2,}', '-', text)       # multiple dashes → one

    # 6. Remove standalone numbers (page refs, section numbers)
    text = re.sub(r'(?<!\w)\d+(?!\w)', '', text)

    return text.strip()

# Apply after building DataFrame
df["text_normalized"] = df["text"].apply(normalize_text)
print("Normalization complete")
print("Sample normalized text:")
print(df["text_normalized"].iloc[0][:300])

Normalization complete
Sample normalized text:
datasheet for contract understanding atticus dataset (cuad) a. who created the dataset (e.g., which team, research group) and on behalf of which entity (e.g. company, institution, organization) the atticus project is a non-profit organization whose mission is to harness the power of ai to accelerate


In [ ]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")

def tokenize(text):
    """
    Word tokenization — standard for legal text
    """
    tokens = nltk.word_tokenize(text)
    return tokens

df["tokens"] = df["text_normalized"].apply(tokenize)
print("Sample tokens:")
print(df["tokens"].iloc[0][:20])

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Sample tokens:
['datasheet', 'for', 'contract', 'understanding', 'atticus', 'dataset', '(', 'cuad', ')', 'a.', 'who', 'created', 'the', 'dataset', '(', 'e.g.', ',', 'which', 'team', ',']


In [ ]:
from nltk.corpus import stopwords
nltk.download("stopwords")

# Standard English stopwords
standard_stopwords = set(stopwords.words("english"))

# Legal terms that MUST be preserved
# Removing these would destroy legal meaning
LEGAL_PRESERVE = {
    # Obligation words
    "shall", "must", "may", "will", "should",
    "not", "no", "never", "neither", "nor",

    # Condition words
    "if", "unless", "until", "except", "provided",
    "without", "including", "notwithstanding",

    # Party references
    "party", "parties", "each", "both", "either",

    # Time references
    "during", "before", "after", "within", "upon",
    "following", "prior",

    # Legal connectors
    "whereas", "hereby", "thereof", "herein",
    "thereto", "hereunder", "hereof"
}

# Final stopword list = standard - legal terms
FINAL_STOPWORDS = standard_stopwords - LEGAL_PRESERVE

def remove_stopwords(tokens):
    return [t for t in tokens if t not in FINAL_STOPWORDS]

df["tokens_clean"] = df["tokens"].apply(remove_stopwords)

# Show what was preserved
sample_removed = set(df["tokens"].iloc[0]) - set(df["tokens_clean"].iloc[0])
print(f"Sample removed stopwords: {list(sample_removed)[:10]}")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Sample removed stopwords: ['the', 'few', 'd', 'own', 'at', 'very', 'which', 'their', 'such', 'that']


In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download("wordnet")
nltk.download("omw-1.4")

# Why lemmatization over stemming for legal text:
# Stemming:      "termination" → "termin"    ← loses meaning
# Lemmatization: "termination" → "terminate" ← keeps meaning

lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

df["tokens_lemma"] = df["tokens_clean"].apply(lemmatize)

print("Before lemmatization:", df["tokens_clean"].iloc[0][:10])
print("After lemmatization: ", df["tokens_lemma"].iloc[0][:10])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Before lemmatization: ['datasheet', 'contract', 'understanding', 'atticus', 'dataset', '(', 'cuad', ')', 'a.', 'created']
After lemmatization:  ['datasheet', 'contract', 'understanding', 'atticus', 'dataset', '(', 'cuad', ')', 'a.', 'created']


In [ ]:
def tokens_to_text(tokens):
    return " ".join(tokens)

df["text_final"] = df["tokens_lemma"].apply(tokens_to_text)

print("Final processed text sample:")
print(df["text_final"].iloc[0][:300])

Final processed text sample:
datasheet contract understanding atticus dataset ( cuad ) a. created dataset ( e.g. , team , research group ) behalf entity ( e.g . company , institution , organization ) atticus project non-profit organization whose mission harness power ai accelerate accurate efficient contract review . atticus pr


In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 17.0 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

# Prepare sentences — Word2Vec needs list of token lists
sentences = df["tokens_lemma"].tolist()

print(f"Training Word2Vec on {len(sentences)} contracts...")

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=200,      # 200-dim embeddings standard for legal
    window=10,            # wider window captures legal context
    min_count=2,          # ignore very rare words
    workers=4,
    epochs=20,            # more epochs = better legal embeddings
    sg=1                  # Skip-gram better for legal domain
)

word2vec_model.save("legal_word2vec.model")
print("Word2Vec model saved!")

# Quick test
test_words = ["agreement", "termination", "confidential", "payment"]
for word in test_words:
    if word in word2vec_model.wv:
        similar = word2vec_model.wv.most_similar(word, topn=3)
        print(f"\n'{word}' most similar: {similar}")

Training Word2Vec on 489 contracts...
Word2Vec model saved!

'agreement' most similar: [('hereof', 0.6609805226325989), ('.', 0.6439275145530701), ('.general', 0.6148512959480286)]

'termination' most similar: [('expiration', 0.8170297145843506), ('survive', 0.7008846402168274), ('terminated', 0.6948466897010803)]

'confidential' most similar: [('information', 0.7336324453353882), ('disclosing', 0.6521665453910828), ('disclosed', 0.625930666923523)]

'payment' most similar: [('payable', 0.7036423683166504), ('paid', 0.683327853679657), ('due', 0.6821971535682678)]


In [ ]:
# Save final clean dataset
df.to_csv("contracts_final.csv", index=False)

# Save just what models need
df[["contract_id", "text_final", "label",
    "confidence", "word_count"]].to_csv(
    "contracts_for_training.csv", index=False
)

print("All files saved!")
print(f"Final dataset shape: {df.shape}")
print(f"\nColumn summary:")
for col in df.columns:
    print(f"  {col}: {df[col].dtype}")

All files saved!
Final dataset shape: (489, 10)

Column summary:
  contract_id: int64
  text: object
  label: object
  confidence: float64
  word_count: int64
  text_normalized: object
  tokens: object
  tokens_clean: object
  tokens_lemma: object
  text_final: object


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("contracts_for_training.csv")

print("Dataset Summary")
print("="*40)
print(f"Total contracts: {len(df)}")
print(f"\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nConfidence stats:")
print(df["confidence"].describe().round(3))

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Class distribution
df["label"].value_counts().plot(
    kind="bar", ax=axes[0],
    title="Contract Type Distribution",
    color="steelblue", edgecolor="black"
)
axes[0].set_xlabel("Contract Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Confidence distribution
df["confidence"].hist(
    bins=20, ax=axes[1],
    title="LegalBERT Confidence Distribution",
    color="green", edgecolor="black"
)
axes[1].set_xlabel("Confidence Score")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig("dataset_overview.png", dpi=150)
plt.show()